# Data transformation

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

train_df = pd.read_csv('train_cleaned.csv')
test_df = pd.read_csv('test_cleaned.csv')

df_all = pd.concat([train_df, test_df], ignore_index=True)

In [8]:
df_all['AgeInMonths_log'] = np.log1p(df_all['AgeInMonths'])

df_all['DateTime'] = pd.to_datetime(df_all['DateTime'])
min_date = df_all['DateTime'].min()
df_all['DaysSinceStart'] = (df_all['DateTime'] - min_date).dt.total_seconds() / (24 * 3600)

scaler = StandardScaler()
df_all[['AgeInMonths_scaled', 'DaysSinceStart_scaled']] = scaler.fit_transform(
    df_all[['AgeInMonths_log', 'DaysSinceStart']]
)

In [9]:
df_transformed = pd.get_dummies(df_all, columns=['AnimalType', 'SexuponOutcome'], drop_first=True)

for col in ['Breed', 'Color']:
    freq_map = df_all[col].value_counts(normalize=True).to_dict()
    df_transformed[f'{col}_freq'] = df_all[col].map(freq_map)

df_transformed['has_name'] = (df_transformed['Name'] != 'No Name').astype(int)

In [12]:
cols_to_drop = ['Name', 'DateTime', 'AgeuponOutcome', 'AgeInMonths', 'AgeInMonths_log', 'DaysSinceStart', 'Breed', 'Color', 'AnimalID']
df_final = df_transformed.drop(columns=[c for c in cols_to_drop if c in df_transformed.columns])

train_transformed = df_final[df_final['OutcomeType'].notnull()].copy()
test_transformed = df_final[df_final['OutcomeType'].isnull()].copy()

train_transformed = train_transformed.drop(columns=['ID'], errors='ignore')
test_transformed = test_transformed.drop(columns=['OutcomeType', 'OutcomeSubtype', 'fold'], errors='ignore')

train_transformed.to_csv('train_transformed.csv', index=False)
test_transformed.to_csv('test_transformed.csv', index=False)

print(f"Transzformált Train alakja: {train_transformed.shape}")
print(f"Transzformált Test alakja: {test_transformed.shape}")

Transzformált Train alakja: (26729, 13)
Transzformált Test alakja: (11456, 11)
